In [6]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [8]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [9]:
# Завдання 3. Ноутбук 02 — вимір валют

# Завдання 3.1. Прочитайте з nbu_raw.raw_rates колонки ingested_at, business_date, payload

nbu_raw_rates_table = f"""
SELECT ingested_at, business_date, payload
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""

raw_rates_df = client.query(nbu_raw_rates_table).to_dataframe()

raw_rates_df

# Завдання 3.2. Розгорніть JSON-текст із payload у колонки.

parsed = pd.json_normalize(raw_rates_df["payload"].map(json.loads))
parsed

# Завдання 3.3. Нормалізуйте значення: 

parsed["cc"] = parsed["cc"].str.strip().str.upper()
parsed["txt"] = parsed["txt"].str.strip()
parsed["r030"] = parsed["r030"].astype("Int64")
parsed

/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cc,exchangedate,r030,rate,special,txt
0,DZD,25.08.2026,12,0.33615,None,Алжирський динар
1,AUD,25.08.2026,36,32.03210,None,Австралійський долар
2,BDT,25.08.2026,50,0.36460,None,Така
3,CAD,25.08.2026,124,32.29020,None,Канадський долар
4,CNY,25.08.2026,156,6.64940,None,Юань Женьміньбі
...,...,...,...,...,...,...
85,PLN,25.08.2026,985,12.10590,None,Злотий
86,XAU,25.08.2026,959,208220.86000,None,Золото
87,XAG,25.08.2026,961,3096.37000,None,Срібло
88,XPT,25.08.2026,962,84163.82000,None,Платина


In [10]:
raw_rates_df

,ingested_at,business_date,payload
0,2026-08-24 16:53:34.648229+00:00,2026-08-24,"{""cc"": ""DZD"", ""exchangedate"": ""25.08.2026"", ""r..."
1,2026-08-24 16:53:34.648247+00:00,2026-08-24,"{""cc"": ""AUD"", ""exchangedate"": ""25.08.2026"", ""r..."
2,2026-08-24 16:53:34.648259+00:00,2026-08-24,"{""cc"": ""BDT"", ""exchangedate"": ""25.08.2026"", ""r..."
3,2026-08-24 16:53:34.648270+00:00,2026-08-24,"{""cc"": ""CAD"", ""exchangedate"": ""25.08.2026"", ""r..."
4,2026-08-24 16:53:34.648280+00:00,2026-08-24,"{""cc"": ""CNY"", ""exchangedate"": ""25.08.2026"", ""r..."
...,...,...,...
85,2026-08-24 18:28:24.698806+00:00,2026-08-24,"{""cc"": ""PLN"", ""exchangedate"": ""25.08.2026"", ""r..."
86,2026-08-24 18:28:24.698822+00:00,2026-08-24,"{""cc"": ""XAU"", ""exchangedate"": ""25.08.2026"", ""r..."
87,2026-08-24 18:28:24.698837+00:00,2026-08-24,"{""cc"": ""XAG"", ""exchangedate"": ""25.08.2026"", ""r..."
88,2026-08-24 18:28:24.698854+00:00,2026-08-24,"{""cc"": ""XPT"", ""exchangedate"": ""25.08.2026"", ""r..."


In [11]:
# Завдання 3.4. Залиште по одному рядку на валюту, узявши найсвіжіший запис за ingested_at

parsed["ingested_at"] = raw_rates_df["ingested_at"].values
parsed["business_date"] = raw_rates_df["business_date"].values

dim = (parsed.sort_values("ingested_at")
          .drop_duplicates(subset="cc", keep="last")).reset_index(drop=True)


dim = dim.rename(columns={
    "cc": "currency_code",
    "txt": "currency_name"
})

dim

,currency_code,exchangedate,r030,rate,special,currency_name,ingested_at,business_date
0,DZD,25.08.2026,12,0.336150,None,Алжирський динар,2026-08-24 18:28:24.698111,2026-08-24
1,AUD,25.08.2026,36,32.032100,None,Австралійський долар,2026-08-24 18:28:24.698141,2026-08-24
2,BDT,25.08.2026,50,0.364600,None,Така,2026-08-24 18:28:24.698161,2026-08-24
3,CAD,25.08.2026,124,32.290200,None,Канадський долар,2026-08-24 18:28:24.698178,2026-08-24
4,CNY,25.08.2026,156,6.649400,None,Юань Женьміньбі,2026-08-24 18:28:24.698193,2026-08-24
5,CZK,25.08.2026,203,2.164300,None,Чеська крона,2026-08-24 18:28:24.698210,2026-08-24
6,DKK,25.08.2026,208,6.977200,None,Данська крона,2026-08-24 18:28:24.698227,2026-08-24
7,HKD,25.08.2026,344,5.704700,None,Гонконгівський долар,2026-08-24 18:28:24.698249,2026-08-24
8,HUF,25.08.2026,348,0.143956,None,Форинт,2026-08-24 18:28:24.698268,2026-08-24
9,INR,25.08.2026,356,0.466920,None,Індійська рупія,2026-08-24 18:28:24.698283,2026-08-24


In [12]:
# Завдання 3.5. Відсортуйте за currency_code і додайте першою колонкою сурогатний ключ currency_key — послідовні числа від 1.

dim.insert(0, "currency_key", range(1, len(dim) +1))

dim

,currency_key,currency_code,exchangedate,r030,rate,special,currency_name,ingested_at,business_date
0,1,DZD,25.08.2026,12,0.336150,None,Алжирський динар,2026-08-24 18:28:24.698111,2026-08-24
1,2,AUD,25.08.2026,36,32.032100,None,Австралійський долар,2026-08-24 18:28:24.698141,2026-08-24
2,3,BDT,25.08.2026,50,0.364600,None,Така,2026-08-24 18:28:24.698161,2026-08-24
3,4,CAD,25.08.2026,124,32.290200,None,Канадський долар,2026-08-24 18:28:24.698178,2026-08-24
4,5,CNY,25.08.2026,156,6.649400,None,Юань Женьміньбі,2026-08-24 18:28:24.698193,2026-08-24
5,6,CZK,25.08.2026,203,2.164300,None,Чеська крона,2026-08-24 18:28:24.698210,2026-08-24
6,7,DKK,25.08.2026,208,6.977200,None,Данська крона,2026-08-24 18:28:24.698227,2026-08-24
7,8,HKD,25.08.2026,344,5.704700,None,Гонконгівський долар,2026-08-24 18:28:24.698249,2026-08-24
8,9,HUF,25.08.2026,348,0.143956,None,Форинт,2026-08-24 18:28:24.698268,2026-08-24
9,10,INR,25.08.2026,356,0.466920,None,Індійська рупія,2026-08-24 18:28:24.698283,2026-08-24


In [13]:
new_row = {
    "currency_key": -1,
    "currency_code": "N/A",
    "currency_name": "Unknown",
    "r030": -1
}
dim.loc[len(dim)] = new_row
dim

/tmp/ipykernel_397/51669522.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dim.loc[len(dim)] = new_row


,currency_key,currency_code,exchangedate,r030,rate,special,currency_name,ingested_at,business_date
0,1,DZD,25.08.2026,12,0.336150,None,Алжирський динар,2026-08-24 18:28:24.698111,2026-08-24
1,2,AUD,25.08.2026,36,32.032100,None,Австралійський долар,2026-08-24 18:28:24.698141,2026-08-24
2,3,BDT,25.08.2026,50,0.364600,None,Така,2026-08-24 18:28:24.698161,2026-08-24
3,4,CAD,25.08.2026,124,32.290200,None,Канадський долар,2026-08-24 18:28:24.698178,2026-08-24
4,5,CNY,25.08.2026,156,6.649400,None,Юань Женьміньбі,2026-08-24 18:28:24.698193,2026-08-24
5,6,CZK,25.08.2026,203,2.164300,None,Чеська крона,2026-08-24 18:28:24.698210,2026-08-24
6,7,DKK,25.08.2026,208,6.977200,None,Данська крона,2026-08-24 18:28:24.698227,2026-08-24
7,8,HKD,25.08.2026,344,5.704700,None,Гонконгівський долар,2026-08-24 18:28:24.698249,2026-08-24
8,9,HUF,25.08.2026,348,0.143956,None,Форинт,2026-08-24 18:28:24.698268,2026-08-24
9,10,INR,25.08.2026,356,0.466920,None,Індійська рупія,2026-08-24 18:28:24.698283,2026-08-24


In [21]:
dim_currency = f"{PROJECT_ID}.nbu_dwh.dim_currency"

dim_to_load = dim[
    ["currency_key", "currency_code", "currency_name", "r030"]
]

schema = [
    bigquery.SchemaField("currency_key", "INT64"),
    bigquery.SchemaField("currency_code", "STRING"),
    bigquery.SchemaField("currency_name", "STRING"),
    bigquery.SchemaField("r030", "INT64"),
]

cfg = bigquery.LoadJobConfig(schema=schema,
                              write_disposition="WRITE_TRUNCATE")


client.load_table_from_dataframe(dim_to_load, dim_currency, job_config=cfg).result()



/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=datalab-504011, location=EU, id=afab4617-6042-4c70-853e-e94e516bcd00>

In [27]:
dim_currency_bq = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
"""

dim_currency_data = client.query(dim_currency_bq).to_dataframe()

/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [54]:
# Завдання 3.8. Перевірте трьома рядками й виведіть результат:
dim_currency_data["currency_code"].is_unique
dim_currency_data["currency_key"].isin([-1]).any()
len(dim_currency_data) == parsed["cc"].nunique() + 1

True